# LSTM Model Visualization Demo

This notebook demonstrates how to use the visualization tools to analyze model predictions and behavior.

In [ ]:
import sys
sys.path.insert(0, '../..')

In [ ]:
import torch
import os
import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display
from visualization import Visualization
from scipy.stats import spearmanr

In [ ]:
visualization = Visualization(
    "/home/campionjp3/Documents/Projects/DeepNetfMRIWorry/checkpoints/lstm_v4_host-smp-n217.crc.pitt.edu_version3/model_best_score=val_loss=0.1113.ckpt",
    "lstmv4vlt"
)

In [ ]:
from datasets.TimeSeriesDataModule import TimeSeriesDataModule
module = TimeSeriesDataModule(subcortical=True, window_timeserie=1190, samples_per_file=1, rate_output=False, identifiers_output=False, batch_size=32, scores_output=True)
module.setup("fit")

In [ ]:
data = pd.DataFrame({
    "population": [],
    "sequence": [],
    "metric": [],
    "condition": [],
    "score": [],
    "rating": [],
    "rating_type": [],
})

In [ ]:
for t in ["training", "block", "full"]:
    m = visualization.visualize_roc_metrics_rates(t)
    for pop in ['raw_not_in_training', 'raw_in_training', 'raw_in_validation', 'validation', 'test']:
        for metric in ["auc", "sensitivity", "specificity", "mae"]:
            if metric == "mae":
                data = pd.concat([data, pd.DataFrame([{
                    "population": pop,
                    "sequence": t,
                    "metric": "mae",
                    "condition": "all",
                    "score": m[i][pop]["mae"],
                    "rating": "all",
                    "rating_type": "all",
                }])], ignore_index=True)
            else:
                for i, condition in enumerate(['Worry_vs_rest', 'Neutral_vs_rest', 'Reappraisal_vs_rest', 'Worry_vs_Neutral', 'Worry_vs_Reappraisal', 'Neutral_vs_Reappraisal']):
                    data = pd.concat([data, pd.DataFrame([{
                        "population": pop,
                        "sequence": t,
                        "metric": metric,
                        "condition": condition,
                        "score": m[i][pop][metric],
                        "rating": "all",
                        "rating_type": "all",
                    }])], ignore_index=True)
            

In [ ]:
for t in ["training", "block", "full"]:
    for threshold in range(5):
        for delta in [False, True]:
            used_threshold = threshold + 1 if delta == False else threshold
            m = visualization.visualize_roc_metrics_rates(t, threshold=used_threshold, asc=2, delta=delta)
            for i, condition in enumerate(['Worry_vs_rest', 'Neutral_vs_rest', 'Reappraisal_vs_rest', 'Worry_vs_Neutral', 'Worry_vs_Reappraisal', 'Neutral_vs_Reappraisal']):
                print(f"Calculating metrics for {t} {condition} with threshold {used_threshold} and delta {delta}")
                for metric in ["auc", "sensitivity", "specificity"]:
                    for pop in ['validation', 'test']:
                        data = pd.concat([data, pd.DataFrame([{
                            "population": pop,
                            "sequence": t,
                            "metric": metric,
                            "condition": condition,
                            "score": m[i][pop][metric],
                            "rating": used_threshold,
                            "rating_type": delta,
                        }])], ignore_index=True)
            

In [ ]:
data.to_csv("../results/metrics.csv")

In [ ]:
shap = np.abs(visualization.shap_test).mean(axis=(0, 2))
data = pd.DataFrame(shap)
data.to_csv("../results/shap_mean.csv")

In [ ]:
shap = np.abs(visualization.shap_test).std(axis=(0, 2))
data = pd.DataFrame(shap)
data.to_csv("../results/shap_std.csv")

In [ ]:
data = np.abs(visualization.shap_test).mean(axis=0)[..., 0].T
data = pd.DataFrame(data)
data.to_csv("../results/shap_data_mean.csv")

In [ ]:
data = np.abs(visualization.shap_test).std(axis=0)[..., 0].T
data = pd.DataFrame(data, )
data.to_csv("../results/shap_data_std.csv")